# Sugarcane Water Requirement Model Comparison

This notebook loads the sugarcane irrigation dataset, trains multiple regression models, compares their performance, and visualizes the results for easier interpretation.

It also shows:
- dataset summary and correlations
- model error comparison
- actual vs predicted values for the best model
- residual distribution
- feature importance when available

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

plt.style.use('seaborn-v0_8') if 'seaborn-v0_8' in plt.style.available else plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

FEATURES = ['temperature', 'humidity', 'rainfall', 'soil_moisture', 'solar_rad', 'crop_stage']
TARGET = 'water_mm_per_week'

def find_existing_path(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find any of: {candidates}')

data_path = find_existing_path([
    'data/sugarcane_dataset.csv',
    'crop-water/data/sugarcane_dataset.csv',
    '../crop-water/data/sugarcane_dataset.csv',
])
model_dir = Path('models') if Path('models').exists() or Path('.').name == 'crop-water' else Path('crop-water/models')
model_dir.mkdir(parents=True, exist_ok=True)

print(f'Data path: {data_path.resolve()}')
print(f'Model directory: {model_dir.resolve()}')

: 

## Load and inspect the data

This cell previews the dataset, checks the target distribution, and gives a quick summary of the feature ranges.

In [ ]:
df = pd.read_csv(data_path)
display(df.head())
print('Shape:', df.shape)
display(df.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=30, color='#2E86AB', edgecolor='white')
axes[0].set_title('Target distribution: weekly water requirement')
axes[0].set_xlabel('Water requirement (mm/week)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df[TARGET], vert=True, patch_artist=True, boxprops=dict(facecolor='#F6C85F'))
axes[1].set_title('Target spread')
axes[1].set_ylabel('Water requirement (mm/week)')
plt.tight_layout()

In [ ]:
corr = df[FEATURES + [TARGET]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title('Feature correlation heatmap')
fig.colorbar(im, ax=ax, shrink=0.85, label='Correlation')
plt.tight_layout()

## Train and compare models

We compare a mix of tree-based and linear models. The metrics below use a train/test split so the comparison is more realistic.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    'Ridge': Ridge(),
    'SVR': SVR(),
}

results = []
trained_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
        'R2': r2_score(y_test, predictions),
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, 'model']
best_model = trained_models[best_model_name]
print(f'Best model by MAE: {best_model_name}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

plot_df = results_df.sort_values('MAE')
axes[0].barh(plot_df['model'], plot_df['MAE'], color='#4E79A7')
axes[0].invert_yaxis()
axes[0].set_title('Model comparison by MAE (lower is better)')
axes[0].set_xlabel('MAE')

axes[1].barh(plot_df['model'], plot_df['R2'], color='#59A14F')
axes[1].invert_yaxis()
axes[1].set_title('Model comparison by R² (higher is better)')
axes[1].set_xlabel('R²')

plt.tight_layout()

## Best model diagnostics

The next plots help interpret how well the best model fits the test set.

In [ ]:
best_predictions = best_model.predict(X_test)
residuals = y_test - best_predictions

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_test, best_predictions, alpha=0.7, color='#8E6C8A')
min_val = min(y_test.min(), best_predictions.min())
max_val = max(y_test.max(), best_predictions.max())
axes[0].plot([min_val, max_val], [min_val, max_val], '--', color='black')
axes[0].set_title(f'Actual vs predicted ({best_model_name})')
axes[0].set_xlabel('Actual water requirement')
axes[0].set_ylabel('Predicted water requirement')

axes[1].hist(residuals, bins=30, color='#F28E2B', edgecolor='white')
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Actual - predicted')
axes[1].set_ylabel('Count')

plt.tight_layout()

diagnostics = pd.DataFrame({
    'actual': y_test.values,
    'predicted': best_predictions,
    'residual': residuals.values,
})
display(diagnostics.head())

In [ ]:
artifact_path = model_dir / f'model_{best_model_name}.joblib'
joblib.dump(best_model, artifact_path)
print(f'Saved best model to {artifact_path.resolve()}')

sample = pd.DataFrame([{
    'temperature': 30.0,
    'humidity': 70.0,
    'rainfall': 10.0,
    'soil_moisture': 20.0,
    'solar_rad': 18.0,
    'crop_stage': 1,
]])
sample_prediction = best_model.predict(sample)[0]
print(f'Sample predicted water requirement: {sample_prediction:.2f} mm/week')

if hasattr(best_model.named_steps['model'], 'feature_importances_'):
    importances = best_model.named_steps['model'].feature_importances_
    importance_df = pd.DataFrame({
        'feature': FEATURES,
        'importance': importances,
    }).sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance_df['feature'], importance_df['importance'], color='#76B7B2')
    ax.set_title(f'Feature importance ({best_model_name})')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    display(importance_df)
else:
    print('Feature importance is not available for this model.')